# Day 56 — The KV cache: what it stores, what it costs, and why prefill and decode are two different machines

Generating a token means looking at every earlier token. Without a cache that means
rerunning the whole model over the whole sequence for every new token. Keeping the keys
and values turns a quadratic loop into a linear one — and splits inference into two
phases with completely different bottlenecks.

In [1]:
import math
import time

import numpy as np

# ================================================================ 1. a small model
def init_params(vocab=24, d_model=64, n_layer=2, n_head=4, ctx=256, seed=0):
    rng = np.random.default_rng(seed)
    s = 0.02
    p = {'tok': rng.standard_normal((vocab, d_model)) * s,
         'pos': rng.standard_normal((ctx, d_model)) * s,
         'g_f': np.ones(d_model), 'b_f': np.zeros(d_model),
         'head': rng.standard_normal((d_model, vocab)) * s}
    for l in range(n_layer):
        p[f'g1_{l}'] = np.ones(d_model)
        p[f'b1_{l}'] = np.zeros(d_model)
        p[f'g2_{l}'] = np.ones(d_model)
        p[f'b2_{l}'] = np.zeros(d_model)
        p[f'Wqkv_{l}'] = rng.standard_normal((d_model, 3 * d_model)) * s
        p[f'Wo_{l}'] = rng.standard_normal((d_model, d_model)) * s
        p[f'W1_{l}'] = rng.standard_normal((d_model, 4 * d_model)) * s
        p[f'W2_{l}'] = rng.standard_normal((4 * d_model, d_model)) * s
    cfg = {'d_model': d_model, 'n_layer': n_layer, 'n_head': n_head, 'ctx': ctx,
           'vocab': vocab}
    return p, cfg


def layer_norm(x, g, b, eps=1e-5):
    mu = x.mean(-1, keepdims=True)
    xc = x - mu
    return xc / np.sqrt((xc ** 2).mean(-1, keepdims=True) + eps) * g + b


def gelu(x):
    return 0.5 * x * (1 + np.tanh(math.sqrt(2 / math.pi) * (x + 0.044715 * x ** 3)))


def softmax(x, axis=-1):
    m = np.max(x, axis=axis, keepdims=True)
    e = np.exp(x - m)
    return e / e.sum(axis=axis, keepdims=True)


def forward_full(p, cfg, ids):
    """The honest version: recompute every position from scratch, every time."""
    T = len(ids)
    H, d = cfg['n_head'], cfg['d_model']
    dh = d // H
    x = p['tok'][ids] + p['pos'][:T]
    mask = np.tril(np.ones((T, T), dtype=bool))
    for l in range(cfg['n_layer']):
        h = layer_norm(x, p[f'g1_{l}'], p[f'b1_{l}'])
        q, k, v = np.split(h @ p[f'Wqkv_{l}'], 3, axis=-1)
        q = q.reshape(T, H, dh).transpose(1, 0, 2)
        k = k.reshape(T, H, dh).transpose(1, 0, 2)
        v = v.reshape(T, H, dh).transpose(1, 0, 2)
        att = np.where(mask, q @ k.transpose(0, 2, 1) / math.sqrt(dh), -np.inf)
        o = (softmax(att, -1) @ v).transpose(1, 0, 2).reshape(T, d)
        x = x + o @ p[f'Wo_{l}']
        h2 = layer_norm(x, p[f'g2_{l}'], p[f'b2_{l}'])
        x = x + gelu(h2 @ p[f'W1_{l}']) @ p[f'W2_{l}']
    return layer_norm(x, p['g_f'], p['b_f']) @ p['head']


# ================================================================ 2. the cache
class KVCache:
    """Two arrays per layer, filled in from the left, plus how much is valid.

    Allocated once at the maximum length: growing a numpy array per token would copy
    the whole thing every step, which is the cache's own version of the quadratic
    problem it exists to solve.
    """

    def __init__(self, cfg, max_len=None):
        self.cfg = cfg
        self.max_len = max_len or cfg['ctx']
        H, dh = cfg['n_head'], cfg['d_model'] // cfg['n_head']
        self.k = [np.zeros((H, self.max_len, dh)) for _ in range(cfg['n_layer'])]
        self.v = [np.zeros((H, self.max_len, dh)) for _ in range(cfg['n_layer'])]
        self.length = 0

    def reset(self):
        self.length = 0

    def append(self, layer, k, v):
        self.k[layer][:, self.length] = k
        self.v[layer][:, self.length] = v

    def bytes(self, dtype_size=2):
        return (2 * self.cfg['n_layer'] * self.cfg['n_head'] *
                (self.cfg['d_model'] // self.cfg['n_head']) * self.length * dtype_size)


def forward_step(p, cfg, cache, token, pos):
    """One new token, reading everything that came before out of the cache."""
    H, d = cfg['n_head'], cfg['d_model']
    dh = d // H
    x = p['tok'][token] + p['pos'][pos]
    for l in range(cfg['n_layer']):
        h = layer_norm(x, p[f'g1_{l}'], p[f'b1_{l}'])
        q, k, v = np.split(h @ p[f'Wqkv_{l}'], 3)
        q = q.reshape(H, dh)
        cache.append(l, k.reshape(H, dh), v.reshape(H, dh))
        n = cache.length + 1
        K = cache.k[l][:, :n]
        V = cache.v[l][:, :n]
        att = np.einsum('hd,hnd->hn', q, K) / math.sqrt(dh)
        o = np.einsum('hn,hnd->hd', softmax(att, -1), V).reshape(d)
        x = x + o @ p[f'Wo_{l}']
        h2 = layer_norm(x, p[f'g2_{l}'], p[f'b2_{l}'])
        x = x + gelu(h2 @ p[f'W1_{l}']) @ p[f'W2_{l}']
    cache.length += 1
    return layer_norm(x, p['g_f'], p['b_f']) @ p['head']


def prefill(p, cfg, cache, ids):
    """Run the prompt through one token at a time to fill the cache.

    A real implementation prefills in one batched pass - the point here is only that
    the cache ends up holding exactly what the full forward would have computed.
    """
    cache.reset()
    out = None
    for i, t in enumerate(ids):
        out = forward_step(p, cfg, cache, int(t), i)
    return out


p, cfg = init_params()
ids = np.random.default_rng(1).integers(0, cfg['vocab'], 64)
cache = KVCache(cfg)
prefill(p, cfg, cache, ids)
print('per layer:', cache.k[0].shape, 'keys and the same for values')
print(f"{cfg['n_layer']} layers, {cache.length} tokens held, {cache.bytes():,} bytes")
print('note what is not kept: the queries - nothing reads them twice')

per layer: (4, 256, 16) keys and the same for values
2 layers, 64 tokens held, 32,768 bytes
note what is not kept: the queries - nothing reads them twice


## The cache is an optimisation, so it must change nothing

The test that matters is a differential one: prefill the prompt, decode token by token,
and require every step to equal what the full forward would have said.

In [2]:
# ================================================================ 3. with and without
def generate_nocache(p, cfg, ids, n, seed=0):
    """Generate n tokens by rerunning the whole model on the whole sequence."""
    rng = np.random.default_rng(seed)
    out = list(ids)
    for _ in range(n):
        logits = forward_full(p, cfg, np.array(out))[-1]
        out.append(int(rng.choice(cfg['vocab'], p=softmax(logits))))
    return out


def generate_cached(p, cfg, ids, n, seed=0):
    rng = np.random.default_rng(seed)
    cache = KVCache(cfg)
    out = list(ids)
    logits = prefill(p, cfg, cache, out)
    for _ in range(n):
        nxt = int(rng.choice(cfg['vocab'], p=softmax(logits)))
        out.append(nxt)
        logits = forward_step(p, cfg, cache, nxt, cache.length)
    return out


def equivalence(p, cfg, ids, n=16):
    """The test that matters: cached decoding must equal the full forward, exactly."""
    cache = KVCache(cfg)
    prefill(p, cfg, cache, ids[:-n])
    worst = 0.0
    seq = list(ids[:-n])
    for t in ids[-n:]:
        step = forward_step(p, cfg, cache, int(t), cache.length)
        seq.append(int(t))
        full = forward_full(p, cfg, np.array(seq))[-1]
        worst = max(worst, float(np.abs(step - full).max()))
    return worst


def timing(p, cfg, prompt_len=64, n=64, seed=0):
    ids = np.random.default_rng(seed).integers(0, cfg['vocab'], prompt_len)
    t0 = time.perf_counter()
    generate_nocache(p, cfg, ids, n)
    t_no = time.perf_counter() - t0
    t0 = time.perf_counter()
    generate_cached(p, cfg, ids, n)
    t_yes = time.perf_counter() - t0
    return t_no, t_yes, t_no / t_yes


print(f'cached step vs full forward, largest difference: {equivalence(p, cfg, ids):.2e}')
print('same sampled tokens both ways:',
      generate_cached(p, cfg, ids[:16], 24) == generate_nocache(p, cfg, ids[:16], 24))
t_no, t_yes, speed = timing(p, cfg, prompt_len=64, n=64)
print(f'64 tokens: {t_no:.2f}s recomputing, {t_yes:.2f}s cached ({speed:.1f}x)')

cached step vs full forward, largest difference: 3.75e-16
same sampled tokens both ways: True
64 tokens: 0.34s recomputing, 0.04s cached (9.6x)


## What it costs, per token and per sequence

Two bytes per number, two arrays, one entry per layer per kv head per token — and one
cache per sequence.

In [3]:
# ================================================================ 4. the arithmetic
def kv_bytes_per_token(n_layer, n_head, head_dim, dtype_size=2, kv_heads=None):
    return 2 * n_layer * (kv_heads or n_head) * head_dim * dtype_size


for name, (L, H, dh, kv) in (('toy model here', (2, 4, 16, None)),
                             ('32 layers, 32 kv heads', (32, 32, 128, None)),
                             ('32 layers, GQA 8 kv heads', (32, 32, 128, 8)),
                             ('80 layers, GQA 8 kv heads', (80, 64, 128, 8))):
    b = kv_bytes_per_token(L, H, dh, 2, kv)
    print(f'{name:>28}: {b:>8,} bytes/token = {b * 131072 / 2**30:6.1f} GB at 128k')

              toy model here:      512 bytes/token =    0.1 GB at 128k
      32 layers, 32 kv heads:  524,288 bytes/token =   64.0 GB at 128k
   32 layers, GQA 8 kv heads:  131,072 bytes/token =   16.0 GB at 128k
   80 layers, GQA 8 kv heads:  327,680 bytes/token =   40.0 GB at 128k


## Prefill and decode

Arithmetic intensity is multiply-accumulates per byte moved. Prefill reuses every loaded
weight across all T tokens; decode uses every loaded number exactly once, which in fp16
is one MAC per two bytes — 0.50, at every context length.

In [4]:
def phase_costs(n_layer, d_model, n_ff, T, dtype_size=2, n_head=32, head_dim=128,
                kv_heads=None):
    """Multiply-accumulates and bytes moved, for a prefill of T tokens and for one
    decode step with T tokens already in the cache.

    The weights have to be read in both cases; the cache has to be read only in
    decode.  Dividing one by the other gives the arithmetic intensity, which is what
    decides whether a phase is limited by the arithmetic units or by memory.
    """
    w = n_layer * (4 * d_model * d_model + 2 * d_model * n_ff)
    pre_flops = T * w + 2 * T * T * d_model * n_layer
    pre_bytes = w * dtype_size + T * d_model * dtype_size
    kv = kv_bytes_per_token(n_layer, n_head, head_dim, dtype_size, kv_heads)
    dec_flops = w + 2 * T * d_model * n_layer
    dec_bytes = w * dtype_size + T * kv
    return {'weights': w, 'prefill_flops': pre_flops, 'prefill_bytes': pre_bytes,
            'prefill_ai': pre_flops / pre_bytes, 'decode_flops': dec_flops,
            'decode_bytes': dec_bytes, 'decode_ai': dec_flops / dec_bytes,
            'kv_per_token': kv}


def batch_decode_time(p, cfg, batches=(1, 8, 32, 128), ctx=512, repeats=7, seed=0):
    """One decode step at several batch sizes.

    Decode reads the same weights whatever the batch size, so until the arithmetic
    catches up, a bigger batch is almost free per sequence.
    """
    rng = np.random.default_rng(seed)
    H, d = cfg['n_head'], cfg['d_model']
    dh = d // H
    rows = []
    for B in batches:
        x = rng.standard_normal((B, d))
        K = rng.standard_normal((B, H, ctx, dh))
        V = rng.standard_normal((B, H, ctx, dh))
        best = math.inf
        for _ in range(repeats):
            t0 = time.perf_counter()
            for l in range(cfg['n_layer']):
                h = layer_norm(x, p[f'g1_{l}'], p[f'b1_{l}'])
                q, k, v = np.split(h @ p[f'Wqkv_{l}'], 3, axis=-1)
                q = q.reshape(B, H, dh)
                att = np.einsum('bhd,bhnd->bhn', q, K) / math.sqrt(dh)
                o = np.einsum('bhn,bhnd->bhd', softmax(att, -1), V).reshape(B, d)
                y = x + o @ p[f'Wo_{l}']
                h2 = layer_norm(y, p[f'g2_{l}'], p[f'b2_{l}'])
                x = y + gelu(h2 @ p[f'W1_{l}']) @ p[f'W2_{l}']
            best = min(best, time.perf_counter() - t0)
        rows.append({'batch': B, 'step_ms': best * 1e3,
                     'per_seq_us': best * 1e6 / B})
    return rows


for T in (128, 512, 2048, 8192):
    c = phase_costs(32, 4096, 16384, T)
    print(f"context {T:>6,}: prefill {c['prefill_ai']:8.1f} MACs/byte, "
          f"decode {c['decode_ai']:.2f}")
print()
for r in batch_decode_time(p, cfg):
    print(f"batch {r['batch']:>4}: {r['step_ms']:6.2f} ms per step, "
          f"{r['per_seq_us']:7.1f} us per sequence")

context    128: prefill     64.3 MACs/byte, decode 0.50
context    512: prefill    261.2 MACs/byte, decode 0.50
context  2,048: prefill   1107.9 MACs/byte, decode 0.50
context  8,192: prefill   5433.0 MACs/byte, decode 0.50

batch    1:   0.32 ms per step,   318.1 us per sequence
batch    8:   0.91 ms per step,   114.0 us per sequence
batch   32:   4.02 ms per step,   125.6 us per sequence
batch  128:  18.10 ms per step,   141.4 us per sequence


## Silent failure: a cache that was not reset

Start a second sequence while the first is still in the cache. Nothing raises; the new
tokens simply attend to someone else's conversation.

In [5]:
# ================================================================ 5. the stale cache
def stale_cache(p, cfg, seed=0, n_first=24, n_second=12):
    """Start a second sequence without clearing the cache.

    Nothing raises: the new tokens simply attend to the previous conversation as well
    as their own, and the logits are finite and normalised and wrong.
    """
    rng = np.random.default_rng(seed)
    first = rng.integers(0, cfg['vocab'], n_first)
    second = rng.integers(0, cfg['vocab'], n_second)
    dirty = KVCache(cfg)
    prefill(p, cfg, dirty, first)                        # conversation one, not reset
    clean = KVCache(cfg)
    gaps, flips = [], 0
    for i, t in enumerate(second):
        ld = forward_step(p, cfg, dirty, int(t), i)
        lc = forward_step(p, cfg, clean, int(t), i)
        gaps.append(float(np.abs(ld - lc).max()))
        flips += int(np.argmax(ld) != np.argmax(lc))
    stale = n_first / (n_first + n_second)
    return {'worst': max(gaps), 'mean': float(np.mean(gaps)), 'flips': flips,
            'steps': len(second), 'stale': stale}


st = stale_cache(p, cfg)
print(f"largest logit difference {st['worst']:.4f} (mean {st['mean']:.4f})")
print(f"argmax changes on {st['flips']} of {st['steps']} steps")
print(f"up to {st['stale'] * 100:.0f}% of what the model could attend to was the "
      f"previous sequence")

largest logit difference 0.2731 (mean 0.1303)
argmax changes on 4 of 12 steps
up to 67% of what the model could attend to was the previous sequence


## The assertions

In [6]:
# ================================================================ 6. assertions
def check():
    p, cfg = init_params()
    ids = np.random.default_rng(1).integers(0, cfg['vocab'], 48)

    # the cache is only an optimisation: same numbers, to the last bit or so
    assert equivalence(p, cfg, ids) < 1e-12

    # and sampling from it gives exactly the same tokens
    assert generate_cached(p, cfg, ids[:16], 24) == generate_nocache(p, cfg, ids[:16], 24)

    # bytes grow linearly with what is in the cache
    cache = KVCache(cfg)
    prefill(p, cfg, cache, ids[:10])
    b10 = cache.bytes()
    prefill(p, cfg, cache, ids[:20])
    assert cache.bytes() == 2 * b10

    # decode has a much lower arithmetic intensity than prefill
    c = phase_costs(32, 4096, 16384, 2048)
    assert c['prefill_ai'] > 100 * c['decode_ai']

    # a stale cache changes the answer without raising anything
    st = stale_cache(p, cfg)
    assert st['worst'] > 1e-3 and st['flips'] > 0
    return True


print('all assertions passed' if check() else 'FAILED')

all assertions passed
